# Migration of ODI Target SQL: Dept_MAp
- Source: ODI_SRC / ODI_TRG
- Conversion Date: 2024-05-24
- Target Platform: Databricks Spark SQL / Delta Lake

In [ ]:
dbutils.widgets.text("v_lst_dt", "")
dbutils.widgets.text("ODI_SESS_NO", "1581fc4b-d7b3-4114-a0d9-e87074762c66")
dbutils.widgets.text("ETL_PROC_WID", "")
dbutils.widgets.text("DATASOURCE_NUM_ID", "")

## ETL Parameters
Fetching the last run date and updating the log table.

In [ ]:
%sql
-- SCEN_TASK_NO {1}: Get last extract date
CREATE OR REPLACE TEMPORARY VIEW v_last_run_date AS
SELECT 
    COALESCE(
        (SELECT date_format(last_run_dt, 'dd-MM-yy') 
         FROM workspace.odi_trg.log_table1 
         WHERE pkg_name = 'Dept_pkg'),
        date_format(current_timestamp(), 'dd-MM-yy')
    ) AS etl_last_dt;

In [ ]:
%sql
-- SCEN_TASK_NO {2}: Update log table
UPDATE workspace.odi_trg.log_table1 
SET last_run_dt = current_timestamp() 
WHERE pkg_name = 'Dept_pkg';

## Staging Table
Handling the C$ staging table.

In [ ]:
%sql
-- SCEN_TASK_NO {30}: Drop staging table
DROP TABLE IF EXISTS workspace.odi_trg.c_0filter;

In [ ]:
%sql
-- SCEN_TASK_NO {40}: Create staging table
CREATE TABLE workspace.odi_trg.c_0filter
(
    DEPARTMENT_ID BIGINT,
    DEPARTMENT_NAME STRING,
    MANAGER_ID BIGINT,
    LOCATION_ID BIGINT,
    LAST_UPD_DT TIMESTAMP
)
USING DELTA;

In [ ]:
%sql
-- SCEN_TASK_NO {50}: Insert into staging
INSERT INTO workspace.odi_trg.c_0filter
(
    DEPARTMENT_ID,
    DEPARTMENT_NAME,
    MANAGER_ID,
    LOCATION_ID,
    LAST_UPD_DT
)
SELECT 
    SRC_DEPARTMENTS.DEPARTMENT_ID,
    SRC_DEPARTMENTS.DEPARTMENT_NAME,
    SRC_DEPARTMENTS.MANAGER_ID,
    SRC_DEPARTMENTS.LOCATION_ID,
    SRC_DEPARTMENTS.LAST_UPD_DT
FROM workspace.odi_src.src_departments AS SRC_DEPARTMENTS
WHERE (SRC_DEPARTMENTS.LAST_UPD_DT >= to_date('${v_lst_dt}', 'dd-MM-yy'));

In [ ]:
%sql
-- SCEN_TASK_NO {60}: Optimize staging table statistics
OPTIMIZE workspace.odi_trg.c_0filter;

## Flow Table
Handling the I$ integration flow table.

In [ ]:
%sql
-- SCEN_TASK_NO {80}: Drop flow table
DROP TABLE IF EXISTS workspace.odi_trg.i_trg_departments_flow;

In [ ]:
%sql
-- SCEN_TASK_NO {90}: Create flow table
CREATE TABLE workspace.odi_trg.i_trg_departments_flow
(
    DEPARTMENT_ID BIGINT,
    DEPARTMENT_NAME STRING,
    MANAGER_ID BIGINT,
    LOCATION_ID BIGINT,
    LAST_UPD_DT TIMESTAMP,
    IND_UPDATE STRING
)
USING DELTA;

In [ ]:
%sql
-- SCEN_TASK_NO {100}: Insert into flow table
INSERT INTO workspace.odi_trg.i_trg_departments_flow
(
    DEPARTMENT_ID,
    DEPARTMENT_NAME,
    MANAGER_ID,
    LOCATION_ID,
    LAST_UPD_DT,
    IND_UPDATE
)
SELECT 
    FILTER_A.DEPARTMENT_ID,
    FILTER_A.DEPARTMENT_NAME,
    FILTER_A.MANAGER_ID,
    FILTER_A.LOCATION_ID,
    FILTER_A.LAST_UPD_DT,
    'I' AS IND_UPDATE
FROM workspace.odi_trg.c_0filter AS FILTER_A;

## Error / Check Tables
Handling audit and error capture tables.

In [ ]:
%sql
-- SCEN_TASK_NO {110}: Create snp_check_tab if not exists
CREATE TABLE IF NOT EXISTS workspace.odi_trg.snp_check_tab
(
    CATALOG_NAME STRING,
    SCHEMA_NAME STRING,
    RESOURCE_NAME STRING,
    FULL_RES_NAME STRING,
    ERR_TYPE STRING,
    ERR_MESS STRING,
    CHECK_DATE TIMESTAMP,
    ORIGIN STRING,
    CONS_NAME STRING,
    CONS_TYPE STRING,
    ERR_COUNT BIGINT
)
USING DELTA;

In [ ]:
%sql
-- SCEN_TASK_NO {120}: Delete previous check stats
DELETE FROM workspace.odi_trg.snp_check_tab
WHERE SCHEMA_NAME = 'ODI_TRG'
AND ORIGIN = '(7)test.Dept_MAp'
AND ERR_TYPE = 'F';

In [ ]:
%sql
-- SCEN_TASK_NO {130}: Create error table if not exists
CREATE TABLE IF NOT EXISTS workspace.odi_trg.e_trg_departments
(
    ODI_ROW_ID STRING,
    ODI_ERR_TYPE STRING,
    ODI_ERR_MESS STRING,
    ODI_CHECK_DATE TIMESTAMP,
    DEPARTMENT_ID BIGINT,
    DEPARTMENT_NAME STRING,
    MANAGER_ID BIGINT,
    LOCATION_ID BIGINT,
    LAST_UPD_DT TIMESTAMP,
    ODI_ORIGIN STRING,
    ODI_CONS_NAME STRING,
    ODI_CONS_TYPE STRING,
    ODI_PK STRING,
    ODI_SESS_NO STRING
)
USING DELTA;

In [ ]:
%sql
-- SCEN_TASK_NO {140}: Clear current session errors
DELETE FROM workspace.odi_trg.e_trg_departments
WHERE (ODI_ERR_TYPE = 'S' AND 1 = 0)
OR (ODI_ERR_TYPE = 'F' AND ODI_ORIGIN = '(7)test.Dept_MAp');

## PK Violation Detection
Checking for primary key uniqueness and null constraints.

In [ ]:
%sql
-- SCEN_TASK_NO {150}: Optimize flow table for uniqueness check
SET spark.databricks.delta.optimize.zorder.checkStatsCollection.enabled = false;
OPTIMIZE workspace.odi_trg.i_trg_departments_flow ZORDER BY (DEPARTMENT_ID);

In [ ]:
%sql
-- SCEN_TASK_NO {160}: Detect PK Violations
INSERT INTO workspace.odi_trg.e_trg_departments
(
    ODI_PK,
    ODI_SESS_NO,
    ODI_ROW_ID,
    ODI_ERR_TYPE,
    ODI_ERR_MESS,
    ODI_ORIGIN,
    ODI_CHECK_DATE,
    ODI_CONS_NAME,
    ODI_CONS_TYPE,
    DEPARTMENT_ID,
    DEPARTMENT_NAME,
    MANAGER_ID,
    LOCATION_ID,
    LAST_UPD_DT
)
SELECT 
    uuid(),
    '${ODI_SESS_NO}', 
    CAST(TRG_DEPARTMENTS.DEPARTMENT_ID AS STRING),
    'F', 
    'ODI-15064: The primary key PK_department_id is not unique.',
    '(7)test.Dept_MAp',
    current_timestamp(),
    'PK_department_id',
    'PK',	
    TRG_DEPARTMENTS.DEPARTMENT_ID,
    TRG_DEPARTMENTS.DEPARTMENT_NAME,
    TRG_DEPARTMENTS.MANAGER_ID,
    TRG_DEPARTMENTS.LOCATION_ID,
    TRG_DEPARTMENTS.LAST_UPD_DT
FROM workspace.odi_trg.i_trg_departments_flow AS TRG_DEPARTMENTS
WHERE EXISTS (
    SELECT 1
    FROM workspace.odi_trg.i_trg_departments_flow AS SUB
    WHERE SUB.DEPARTMENT_ID = TRG_DEPARTMENTS.DEPARTMENT_ID
    GROUP BY SUB.DEPARTMENT_ID
    HAVING COUNT(1) > 1
);

In [ ]:
%sql
-- SCEN_TASK_NO {170}: Detect Not Null Violations
INSERT INTO workspace.odi_trg.e_trg_departments
(
    ODI_PK,
    ODI_SESS_NO,
    ODI_ROW_ID,
    ODI_ERR_TYPE,
    ODI_ERR_MESS,
    ODI_CHECK_DATE,
    ODI_ORIGIN,
    ODI_CONS_NAME,
    ODI_CONS_TYPE,
    DEPARTMENT_ID,
    DEPARTMENT_NAME,
    MANAGER_ID,
    LOCATION_ID,
    LAST_UPD_DT
)
SELECT
    uuid(),
    '${ODI_SESS_NO}', 
    CAST(DEPARTMENT_ID AS STRING),
    'F', 
    'ODI-15066: The column DEPARTMENT_NAME cannot be null.',
    current_timestamp(),
    '(7)test.Dept_MAp',
    'DEPARTMENT_NAME',
    'NN',	
    DEPARTMENT_ID,
    DEPARTMENT_NAME,
    MANAGER_ID,
    LOCATION_ID,
    LAST_UPD_DT
FROM workspace.odi_trg.i_trg_departments_flow
WHERE DEPARTMENT_NAME IS NULL;

In [ ]:
%sql
-- SCEN_TASK_NO {180}: Optimize Error table
SET spark.databricks.delta.optimize.zorder.checkStatsCollection.enabled = false;
OPTIMIZE workspace.odi_trg.e_trg_departments ZORDER BY (DEPARTMENT_ID);

In [ ]:
%sql
-- SCEN_TASK_NO {190}: Remove records with errors from flow table
MERGE INTO workspace.odi_trg.i_trg_departments_flow AS T
USING (
    SELECT DISTINCT DEPARTMENT_ID 
    FROM workspace.odi_trg.e_trg_departments 
    WHERE ODI_SESS_NO = '${ODI_SESS_NO}'
) AS E
ON T.DEPARTMENT_ID = E.DEPARTMENT_ID
WHEN MATCHED THEN DELETE;

In [ ]:
%sql
-- SCEN_TASK_NO {200}: Log Error Summary
INSERT INTO workspace.odi_trg.snp_check_tab
(
    SCHEMA_NAME,
    RESOURCE_NAME,
    FULL_RES_NAME,
    ERR_TYPE,
    ERR_MESS,
    CHECK_DATE,
    ORIGIN,
    CONS_NAME,
    CONS_TYPE,
    ERR_COUNT
)
SELECT	
    'ODI_TRG',
    'TRG_DEPARTMENTS',
    'ODI_TRG.TRG_DEPARTMENTS',
    E.ODI_ERR_TYPE,
    E.ODI_ERR_MESS,
    E.ODI_CHECK_DATE,
    E.ODI_ORIGIN,
    E.ODI_CONS_NAME,
    E.ODI_CONS_TYPE,
    COUNT(1) 
FROM workspace.odi_trg.e_trg_departments AS E
WHERE E.ODI_ERR_TYPE = 'F'
AND E.ODI_ORIGIN = '(7)test.Dept_MAp'
GROUP BY E.ODI_ERR_TYPE, E.ODI_ERR_MESS, E.ODI_CHECK_DATE, E.ODI_ORIGIN, E.ODI_CONS_NAME, E.ODI_CONS_TYPE;

## Mark Records for Update
Flagging records as Update (U), No Change (N), or Insert (I).

In [ ]:
%sql
-- SCEN_TASK_NO {210}: Flag for update if key exists
MERGE INTO workspace.odi_trg.i_trg_departments_flow AS T
USING workspace.odi_trg.trg_departments AS S
ON T.DEPARTMENT_ID = S.DEPARTMENT_ID
WHEN MATCHED THEN UPDATE SET T.IND_UPDATE = 'U';

In [ ]:
%sql
-- SCEN_TASK_NO {220}: Flag as 'No Change' if all non-key attributes match
MERGE INTO workspace.odi_trg.i_trg_departments_flow AS T
USING workspace.odi_trg.trg_departments AS S
ON T.DEPARTMENT_ID = S.DEPARTMENT_ID
AND (T.DEPARTMENT_NAME = S.DEPARTMENT_NAME OR (T.DEPARTMENT_NAME IS NULL AND S.DEPARTMENT_NAME IS NULL)) 
AND (T.MANAGER_ID = S.MANAGER_ID OR (T.MANAGER_ID IS NULL AND S.MANAGER_ID IS NULL)) 
AND (T.LOCATION_ID = S.LOCATION_ID OR (T.LOCATION_ID IS NULL AND S.LOCATION_ID IS NULL)) 
AND (T.LAST_UPD_DT = S.LAST_UPD_DT OR (T.LAST_UPD_DT IS NULL AND S.LAST_UPD_DT IS NULL))
WHEN MATCHED THEN UPDATE SET T.IND_UPDATE = 'N';

## MERGE into Target
Executing the final synchronization with the target table.

In [ ]:
%sql
-- SCEN_TASK_NO {230} + {240}: Combined MERGE into TRG_DEPARTMENTS
MERGE INTO workspace.odi_trg.trg_departments AS T
USING workspace.odi_trg.i_trg_departments_flow AS S
ON T.DEPARTMENT_ID = S.DEPARTMENT_ID
WHEN MATCHED AND S.IND_UPDATE = 'U' THEN UPDATE SET 
    T.DEPARTMENT_NAME = S.DEPARTMENT_NAME,
    T.MANAGER_ID = S.MANAGER_ID,
    T.LOCATION_ID = S.LOCATION_ID,
    T.LAST_UPD_DT = S.LAST_UPD_DT
WHEN NOT MATCHED AND S.IND_UPDATE = 'I' THEN INSERT 
(
    DEPARTMENT_ID,
    DEPARTMENT_NAME,
    MANAGER_ID,
    LOCATION_ID,
    LAST_UPD_DT 
) 
VALUES 
(
    S.DEPARTMENT_ID,
    S.DEPARTMENT_NAME,
    S.MANAGER_ID,
    S.LOCATION_ID,
    S.LAST_UPD_DT
);

## Cleanup
Removing temporary staging and flow tables.

In [ ]:
%sql
-- SCEN_TASK_NO {280}: Drop staging table
DROP TABLE IF EXISTS workspace.odi_trg.c_0filter;

In [ ]:
%sql
-- SCEN_TASK_NO {300}: Drop flow table
DROP TABLE IF EXISTS workspace.odi_trg.i_trg_departments_flow;